# Interpretability of the final model with SHAP

Global interpretability analysis of the Gradient Boosting model using SHAP values.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")

shap.__version__

## Loading training data and final model

In [ ]:
X_train = pd.read_parquet("../data/dataset/X_train.parquet")
y_train = pd.read_parquet("../data/dataset/y_train.parquet").squeeze()

X_train.shape, y_train.shape, y_train.value_counts(normalize=True)

In [ ]:
final_model = joblib.load("../models/final_diabetes_model.pkl")
final_model

# El modelo está dentro del diccionario
real_model = final_model["model"]

real_model

## Explainer SHAP

In [ ]:
# Muestra para acelerar el cálculo
X_train_sample = X_train.sample(n=min(2000, len(X_train)), random_state=42)

# SHAP necesita una función callable → usamos predict_proba
explainer = shap.Explainer(
    real_model.predict_proba,
    X_train_sample
)

# Cálculo de valores SHAP
shap_values = explainer(X_train_sample)

# Para clasificación binaria → clase positiva (índice 1)
shap_values_pos = shap_values.values[:, :, 1]

shap_values_pos.shape

## Global importance (bar plot SHAP)

In [ ]:
shap.summary_plot(
    shap_values_pos,
    X_train_sample,
    plot_type="bar",
    show=False
)

plt.title("Importancia global de variables (SHAP)")
plt.tight_layout()
plt.show()

## SHAP summary plot (beeswarm)

In [ ]:
shap.summary_plot(
    shap_values_pos,
    X_train_sample,
    show=False
)

plt.title("SHAP summary plot - Modelo final")
plt.tight_layout()
plt.show()